# Sozo BioLattice: Multimodal Pathogen Persistence Predictor

**Fine-tuning Gemma-4-E2B for Climate-Pathogen Surface Interaction Analysis**

This notebook implements a state-of-the-art vision-language fine-tuning pipeline using [Unsloth](https://github.com/unslothai/unsloth) and Google's Gemma-4-E2B model. The goal is to predict pathogen persistence (`T90`, `lambda`) on material surfaces as a function of climate reanalysis data (temperature, humidity, dewpoint) and surface imagery.

## Objectives
- Ingest high-resolution climate reanalysis (ERA5-Land) for the SADC region
- Synthesize pathogen survival kinetics from peer-reviewed literature (CORD-19)
- Fine-tune a 4-bit quantized vision-language model for edge deployment
- Export merged weights for UNICEF Climate Ventures / mobile triage applications

## References
- Starter notebook: [Gemma-4 3.1B Unsloth](https://www.kaggle.com/code/danielhanchen/gemma4-31b-unsloth)
- [ECMWF ERA5-Land Hourly Data](https://www.kaggle.com/datasets/ecmwf/era5-land-release)
- [CORD-19 Research Dataset](https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge)

## 1. Environment Setup

Install the required packages for Unsloth, climate data processing, and dataset handling.

In [ ]:
# Install core dependencies
!pip install -q unsloth xarray netCDF4 datasets trl transformers accelerate

## 2. Imports

Load all necessary libraries for model initialization, data ingestion, and training.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import xarray as xr
import pandas as pd
import numpy as np
from PIL import Image

from unsloth import FastVisionModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 3. Model Initialization (Edge-Native Variant)

Load the **Gemma-4-E2B** model in 4-bit quantization for efficient fine-tuning. We apply LoRA/PEFT targeting vision-language projector layers alongside attention projections to enable cross-modal learning between surface imagery and climate text.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `model_name` | `unsloth/gemma-4-e2b-bnb-4bit` | Edge-optimized 3.1B vision-language model |
| `r` | 32 | LoRA rank |
| `target_modules` | q/k/v/o_proj + mm_projector | Attention + multimodal projector |
| `max_seq_length` | 4096 | Context window |

In [ ]:
# Configuration
MAX_SEQ_LENGTH = 4096
MODEL_NAME = "unsloth/gemma-4-e2b-bnb-4bit"

# 3.1 Load base model with 4-bit quantization
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
)

# 3.2 Apply PEFT/LoRA for efficient fine-tuning
model = FastVisionModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "mm_projector"],
    use_gradient_checkpointing="unsloth",
)

print("Model loaded successfully.")
print(f"Trainable parameters: {model.count_parameters():,}")

## 4. Data Ingestion: Climate + Pathogen Bio-Logic

This section synchronizes two distinct data streams:

1. **Climate Reanalysis (ERA5-Land)**: High-resolution meteorological data (temperature `t2m`, dewpoint `d2m`, relative humidity) for Zimbabwe/SADC region.
2. **Pathogen Durability (CORD-19)**: Peer-reviewed abstracts filtered for persistence, half-life, and stability studies.

> **Note**: Update the dataset paths below to match your Kaggle environment or local file system.

In [ ]:
# --- CONFIGURE YOUR DATA PATHS HERE ---
ERA5_PATH = '/kaggle/input/era5-land-release/era5_2026_zimbabwe.nc'
CORD19_PATH = '/kaggle/input/cord-19-research-challenge/metadata.csv'
SURFACE_IMAGE_PATH = '/kaggle/input/sozo-lab-surfaces/sample.jpg'  # Replace with your lab imagery

def prepare_sozo_biolattice_data(era5_path: str, cord19_path: str, n_samples: int = 1000):
    """
    Synchronize climate reanalysis with pathogen persistence literature.
    
    Args:
        era5_path: Path to ERA5-Land NetCDF file
        cord19_path: Path to CORD-19 metadata CSV
        n_samples: Number of samples to extract from each source
    
    Returns:
        Merged DataFrame with climate physics + bio-logic annotations
    """
    # 4.1 Load Climate Reanalysis (ERA5-Land - Zimbabwe Slice)
    # Provides the 'Guti' weather logic for SADC region
    print(f"[1/3] Loading ERA5-Land data from: {era5_path}")
    ds = xr.open_dataset(era5_path)
    climate_df = ds.to_dataframe().reset_index()
    print(f"      Climate records: {len(climate_df):,}")
    
    # 4.2 Load Pathogen Durability (CORD-19 / GWPP Metadata)
    # Extracts surface persistence variables (P-SURV logic)
    print(f"[2/3] Loading CORD-19 metadata from: {cord19_path}")
    persist_df = pd.read_csv(cord19_path)
    
    # Filter for persistence-specific studies (SOTA logic)
    keywords = ['half-life', 'persistence', 'stability', 'survival', 'decay']
    pattern = '|'.join(keywords)
    mask = persist_df['abstract'].str.contains(pattern, case=False, na=False)
    persist_df = persist_df[mask].copy()
    print(f"      Persistence studies found: {len(persist_df):,}")
    
    # 4.3 Cross-reference: Surface Imagery + Bio-Logic
    # For baseline, use abstract content as the 'Clinical Truth'
    print("[3/3] Merging climate and bio-logic datasets...")
    merged = pd.merge(
        climate_df.head(n_samples),
        persist_df.head(n_samples),
        left_index=True,
        right_index=True,
        how='inner'
    )
    
    # Add computed features
    if 't2m' in merged.columns:
        merged['t2m_celsius'] = merged['t2m'] - 273.15  # K -> °C
    if 'd2m' in merged.columns and 't2m' in merged.columns:
        # Approximate relative humidity from dewpoint depression
        merged['rh_approx'] = 100 - 5 * (merged['t2m'] - merged['d2m'])
        merged['rh_approx'] = merged['rh_approx'].clip(0, 100)
    
    print(f"      Final merged samples: {len(merged):,}")
    return merged

# Execute data preparation
data = prepare_sozo_biolattice_data(ERA5_PATH, CORD19_PATH, n_samples=1000)
data.head()

## 5. Multimodal Prompt Formatting

Construct fine-tuning prompts that align **visual porosity** (surface imagery) with **climate reanalysis** (text) and **pathogen persistence** (target reasoning). The Gemma chat template uses `<start_of_turn>` tokens to delimit user and model turns.

### Prompt Structure
```
<start_of_turn>user
<image>
Material Identification: {journal} Research Context
Climate Reanalysis: Temp {t2m}K, Dewpoint {d2m}K
Pathogen: Predict persistence based on environmental markers.<end_of_turn>
<start_of_turn>model
Persistence Logic: {abstract[:300]}...
Vulnerability Score: HIGH (Surface-Climate intersection).<end_of_turn>
```

In [ ]:
def format_prompt(row: pd.Series, image_path: str) -> dict:
    """
    Format a single row into a multimodal training example.
    
    Args:
        row: Merged DataFrame row with climate + bio fields
        image_path: Path to surface morphology image
    
    Returns:
        dict with 'text' (prompt) and 'image' (path or PIL.Image)
    """
    # Extract fields with safe defaults
    temp_k = row.get('t2m', 298.0)
    dew_k = row.get('d2m', 293.0)
    journal = row.get('journal', 'Unknown Journal')
    abstract = str(row.get('abstract', 'No abstract available.'))[:300]
    
    # Build instruction (user turn)
    instruction = (
        f"Material Identification: {journal} Research Context\n"
        f"Climate Reanalysis: Temp {temp_k:.1f}K, Dewpoint {dew_k:.1f}K\n"
        f"Pathogen: Predict persistence based on these environmental markers."
    )
    
    # Build response (model turn)
    response = (
        f"Persistence Logic: {abstract}... \n"
        f"Vulnerability Score: HIGH (Due to Surface-Climate intersection)."
    )
    
    # Assemble full prompt with Gemma chat template
    full_text = (
        f"<start_of_turn>user\n"
        f"<image>\n"
        f"{instruction}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{response}<end_of_turn>"
    )
    
    # Load image if available; otherwise None
    if os.path.exists(image_path):
        image = Image.open(image_path).convert('RGB')
    else:
        image = None
    
    return {
        "text": full_text,
        "image": image if image is not None else image_path
    }

# Test formatting on first row
sample = format_prompt(data.iloc[0], SURFACE_IMAGE_PATH)
print("=== SAMPLE PROMPT ===")
print(sample['text'][:800] + "...")
print(f"\nImage: {sample['image']}")

## 6. HuggingFace Dataset Preparation

Convert the pandas DataFrame into a `datasets.Dataset` object and apply the formatting function across all rows. This produces the final training corpus for the SFTTrainer.

In [ ]:
# 6.1 Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(data, preserve_index=False)
print(f"Dataset created: {len(train_ds)} rows")
print(f"Column names: {train_ds.column_names}")

# 6.2 Map formatting function
# Note: For large datasets, consider batched mapping or streaming
train_ds = train_ds.map(
    lambda row: format_prompt(row, SURFACE_IMAGE_PATH),
    remove_columns=train_ds.column_names,  # Drop original columns
)

print(f"\nFormatted dataset: {len(train_ds)} examples")
print(f"Example keys: {train_ds[0].keys()}")

# 6.3 Inspect a formatted example
print("\n=== FORMATTED EXAMPLE ===")
print(train_ds[0]['text'][:1000])

## 7. Training Configuration

Define hyperparameters optimized for edge deployment with limited GPU memory.

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `per_device_train_batch_size` | 2 | Fits 4-bit model in ~16GB VRAM |
| `gradient_accumulation_steps` | 4 | Effective batch = 8 |
| `max_steps` | 100 | One-shot/POC training |
| `learning_rate` | 5e-5 | Standard LoRA LR |
| `warmup_steps` | 10 | Stable early training |
| `optim` | `adamw_8bit` | Memory-efficient optimizer |
| `fp16` | True | Mixed precision speedup |

In [ ]:
# 7.1 Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,
    learning_rate=5e-5,
    fp16=True,
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=50,
    output_dir="/kaggle/working/sozo_biolattice_weights",
    report_to="none",  # Disable WandB/TensorBoard for Kaggle
)

# 7.2 Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print("Trainer initialized.")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: {training_args.max_steps}")

## 8. Fine-Tuning Execution

Run the supervised fine-tuning loop. With 100 steps and an effective batch size of 8, this completes ~800 training examples — sufficient for a one-shot proof-of-concept on climate-pathogen alignment.

In [ ]:
# 8.1 Train
trainer_stats = trainer.train()

# 8.2 Print summary
print("\n=== TRAINING COMPLETE ===")
print(f"Final loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics.get('train_runtime', 0):.1f}s")

## 9. Export & Inference

### 9.1 Model Export

Save the merged model in 16-bit for downstream deployment (UNICEF Climate Ventures, mobile triage, edge devices). The `merged_16bit` method fuses LoRA weights back into the base model for easier inference.

In [ ]:
# Export merged model (16-bit) for production inference
EXPORT_PATH = "sozo_biolattice_v1"

model.save_pretrained_merged(
    EXPORT_PATH,
    tokenizer,
    save_method="merged_16bit"
)

print(f"Model exported to: {EXPORT_PATH}")
print(f"Files: {os.listdir(EXPORT_PATH)[:5]} ...")

### 9.2 Sample Inference

Test the fine-tuned model on a real-world prompt: **Vibrio cholerae** persistence under Harare-like conditions (32°C, 85% humidity).

In [ ]:
# Prepare a test prompt
test_prompt = (
    "<start_of_turn>user\n"
    "<image>\n"
    "Environmental Context: Temp 32°C, Humidity 85% (Harare Forecast).\n"
    "Pathogen: Vibrio cholerae.\n"
    "Analyze material-pathogen interaction and predict T90.<end_of_turn>\n"
    "<start_of_turn>model\n"
)

# Load a test image (or reuse training image)
test_image = Image.open(SURFACE_IMAGE_PATH).convert('RGB') if os.path.exists(SURFACE_IMAGE_PATH) else None

# Tokenize and generate
inputs = tokenizer(
    text=test_prompt,
    images=test_image,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("=== MODEL RESPONSE ===")
print(response[len(test_prompt):])

## 10. Summary & Next Steps

| Component | Status | Notes |
|-----------|--------|-------|
| Model | Gemma-4-E2B 3.1B (4-bit) | Edge-optimized VLM |
| Fine-tuning | LoRA r=32 | Attention + projector layers |
| Climate Data | ERA5-Land | Zimbabwe/SADC slice |
| Bio-Logic | CORD-19 filtered | Persistence abstracts |
| Export | Merged 16-bit | Ready for mobile/edge |

### Recommended Next Steps
1. **Replace placeholder image path** with actual lab-captured surface morphology photos.
2. **Expand CORD-19 filtering** with structured extraction of `T90` and `lambda` values from full-text articles.
3. **Add validation split** (~10%) to monitor overfitting during longer training runs.
4. **Quantize to GGUF/Q4_K_M** for ultra-low-power edge deployment.
5. **Integrate real-time ERA5 API** for dynamic climate-pathogen risk mapping.

---

*Sozo BioLattice — Built for the Kaggle Gemma 4 Good Hackathon & UNICEF Climate Ventures*